In [1]:
import numpy as np


class OrderedTargetEncoder:
  """Ordered Target Statistic encoding to prevent target leakage."""

  def __init__(self, prior_weight=1.0):
    self.prior_weight = prior_weight
    self.global_prior = 0.0

  def fit_transform(self, categories, y):
    n_samples = len(categories)
    self.global_prior = np.mean(y)

    # Step 1: Create a random permutation sequence
    perm = np.random.permutation(n_samples)
    inv_perm = np.empty_like(perm)
    inv_perm[perm] = np.arange(n_samples)

    cats_perm = categories[perm]
    y_perm = y[perm]

    # Step 2: Compute historical cumulative targets
    cat_counts = {}
    cat_target_sums = {}
    encoded_perm = np.zeros(n_samples, dtype=float)

    for i in range(n_samples):
      c = cats_perm[i]
      count = cat_counts.get(c, 0)
      target_sum = cat_target_sums.get(c, 0.0)

      # Ordered Statistic: prior smoothing
      encoded_perm[i] = (target_sum + self.prior_weight * self.global_prior) / (
          count + self.prior_weight
      )

      # Update running totals
      cat_counts[c] = count + 1
      cat_target_sums[c] = target_sum + y_perm[i]

    # Return values in original sample order
    return encoded_perm[inv_perm]


class ObliviousDecisionTree:
  """Symmetric decision tree where the exact same split criterion applies

  to all nodes at the same depth level.
  """

  def __init__(self, max_depth=3):
    self.max_depth = max_depth
    self.split_features = []
    self.split_thresholds = []
    self.leaf_values = None

  def fit(self, X, y):
    n_samples, n_features = X.shape
    current_leaf_indices = np.zeros(n_samples, dtype=int)

    self.split_features = []
    self.split_thresholds = []

    for depth in range(self.max_depth):
      best_var_reduction = -1.0
      best_f, best_th = 0, 0.0
      n_leaves = 1 << depth

      # Scan across all features to find a unified level split
      for f in range(n_features):
        thresholds = np.unique(X[:, f])
        for th in thresholds:
          # Compute variance reduction across all current leaf partitions
          var_reduction = 0.0
          for leaf in range(n_leaves):
            mask = current_leaf_indices == leaf
            if np.sum(mask) < 2:
              continue

            y_subset = y[mask]
            parent_var = np.var(y_subset) * len(y_subset)

            left = mask & (X[:, f] <= th)
            right = mask & (X[:, f] > th)

            if np.sum(left) == 0 or np.sum(right) == 0:
              continue

            left_var = np.var(y[left]) * np.sum(left)
            right_var = np.var(y[right]) * np.sum(right)
            var_reduction += parent_var - (left_var + right_var)

          if var_reduction > best_var_reduction:
            best_var_reduction = var_reduction
            best_f = f
            best_th = th

      self.split_features.append(best_f)
      self.split_thresholds.append(best_th)

      # Update leaf index with bitwise shifts
      bit = (X[:, best_f] > best_th).astype(int)
      current_leaf_indices = (current_leaf_indices << 1) | bit

    # Compute optimal output values for each symmetric leaf
    total_leaves = 1 << self.max_depth
    self.leaf_values = np.zeros(total_leaves)
    for leaf in range(total_leaves):
      mask = current_leaf_indices == leaf
      if np.sum(mask) > 0:
        self.leaf_values[leaf] = np.mean(y[mask])

  def predict(self, X):
    n_samples = X.shape[0]
    leaf_indices = np.zeros(n_samples, dtype=int)

    # Fast bitwise evaluation with zero branching
    for f, th in zip(self.split_features, self.split_thresholds):
      bit = (X[:, f] > th).astype(int)
      leaf_indices = (leaf_indices << 1) | bit

    return self.leaf_values[leaf_indices]


# =========================================================================
# DEMO EXECUTION
# =========================================================================
if __name__ == '__main__':
  np.random.seed(42)

  # Synthetic data with categorical labels and continuous features
  categories = np.array(
      ['A', 'B', 'B', 'A', 'C', 'A', 'C', 'B', 'C', 'A'] * 10
  )
  X_continuous = np.random.randn(100, 2)
  y = (categories == 'B').astype(float) * 2.0 + X_continuous[:, 0] * 1.5

  # 1. Transform categories using Ordered Target Encoding
  encoder = OrderedTargetEncoder(prior_weight=1.0)
  encoded_cat = encoder.fit_transform(categories, y).reshape(-1, 1)

  # 2. Combine features
  X = np.hstack([X_continuous, encoded_cat])

  # 3. Fit Symmetric Oblivious Tree
  tree = ObliviousDecisionTree(max_depth=3)
  tree.fit(X, y)
  preds = tree.predict(X)

  mse = np.mean((y - preds) ** 2)
  print('=' * 60)
  print('  CATBOOST SIMULATION (ORDERED TARGET STATS & OBLIVIOUS TREES)')
  print('=' * 60)
  print(f'Total Samples Processed      : {X.shape[0]}')
  print(f'Tree Depth (Symmetric Splits): {tree.max_depth}')
  print(f'Total Oblivious Leaves       : {1 << tree.max_depth}')
  print(f'Training MSE                 : {mse:.6f}')
  print('=' * 60)

  CATBOOST SIMULATION (ORDERED TARGET STATS & OBLIVIOUS TREES)
Total Samples Processed      : 100
Tree Depth (Symmetric Splits): 3
Total Oblivious Leaves       : 8
Training MSE                 : 0.340022
